# FFT spectrum firmware data analysis
This part of the notebook analyzes the data obtained at the **ACCUMULATOR_1** core in the diagram above.

System description:

- ADC resolution: 12 bits (signed, native range ±2048)
- **Full-scale (FS) in FPGA:** 2^(15) = 32,768 (due to MSB alignment into 16-bit register)
- Sampling rate Fs = 32 MHz
- FFT size N = 65536
- FFT has NO internal scaling
- FFT IP outputs averaged power per bin:

\begin{equation}
    P[k] = average( |X[k]|^2 )
  \end{equation}

- Signal is complex (IQ)
- Tone has been digitally mixed to DC

## Important Notice: MSB Alignment in 16-bit Registers

Since the 12-bit data in the RFSoC ADC is MSB-aligned in a 16-bit register, important implications follow for all downstream calculations. 

**For detailed explanation and derivation, see:** [00_zcu111_calibration.ipynb - Appendix: MSB Alignment](./00_zcu111_calibration.ipynb)

**Key Points for FFT Analysis:**
* 12-bit ADC value is left-shifted by 4 bits in 16-bit FPGA registers
* Effective Full-Scale: **FS = 2^15 = 32,768** (NOT 2^11 = 2,048)
* FFT processes data at this 16-bit scale throughout
* This introduces a **+24 dB offset** in all power calculations (20·log₁₀(16))
* All formulas in this notebook use **FS = 32,768**

---

# Notebook Section 2 — Mathematical Background

## 1. What does the FFT compute?

Since the FFT is unscaled:

$$
\begin{equation}
X[k] = \sum_{n=0}^{N-1} x[n] e^{-j2\pi kn/N}
\end{equation}
$$

For a coherent complex tone (tone whose frequency is exactly aligned with a bin of the FFT):

$$
\begin{equation}
|X[k_0]| = N \cdot A
\end{equation}
$$

Where:

* $( A )$ = actual tone amplitude in ADC LSB units.

Therefore:

$$
\begin{equation}
A = \frac{|X[k_0]|}{N}
\end{equation}
$$

---

## 2. What does the IP give us?
The IP (FFT + Accumulator) gives:

$$
\begin{equation}
P[k] = |X[k]|^2
\end{equation}
$$

So:

$$
\begin{equation}
|X[k]| = \sqrt{P[k]}
\end{equation}
$$

Therefore:

$$
\begin{equation}
A = \frac{\sqrt{P[k]}}{N}
\end{equation}
$$

---

## 3. Convert to dBFS

Full-scale amplitude:

$$
\begin{equation}
FS = 2^{15} = 32768
\end{equation}
$$

Normalized amplitude:

$$
\begin{equation}
A_{FS} = \frac{A}{FS}
\end{equation}
$$

Amplitude in dBFS:

$$
\begin{equation}
20\log_{10}(A_{FS})
\end{equation}
$$

Expected value for half-scale tone:

$$
\begin{equation}
20\log_{10}(0.5) = -6.02 \text{ dBFS}
\end{equation}
$$

---

## Definition of dBFS in this System

In this system, **0 dBFS corresponds to a full-scale complex sinusoid at the ADC input**.

For a signed (N)-bit ADC:

$$
FS = 2^{N-1}
$$

Because the RFSoC ADC data is **MSB-aligned into a 16-bit bus**, the effective full-scale used by the FPGA logic becomes:

$$
FS_{FPGA} = 2^{15} = 32768
$$

Therefore:

* **0 dBFS** → amplitude = 32768
* **−6.02 dBFS** → amplitude = 16384
* **−20 dBFS** → amplitude ≈ 3276

This reference is used for all amplitude and power calculations inside the DSP chain.

## RF Power Calibration

The FFT outputs amplitudes referenced to **digital full-scale (dBFS)**.
However, RF measurements require power referenced to **1 mW (dBm)**.

To convert between both domains, a calibration constant is introduced.

$$
P_{RF}(dBm) = P_{FFT}(dBFS) + C
$$

Where:

* ($P_{FFT}$(dBFS)) = power measured from the FFT
* ($C$) = system calibration constant

This constant accounts for:

• ADC analog input scaling
• RF front-end attenuation
• DSP gain and quantization effects
• impedance conversion ($50\,\Omega$)

---

### Example (Measured Calibration)

From experimental measurements:

$$
C \approx 39.1\,\text{dB}
$$

Therefore:

$$
P_{RF}(dBm) = P_{FFT}(dBFS) + 39.1
$$

Example:

| FFT Peak | RF Power  |
| -------- | --------- |
| -30 dBFS | +9.1 dBm  |
| -40 dBFS | -0.9 dBm  |
| -50 dBFS | -10.9 dBm |

---

## Spectral Resolution

The FFT bin width is determined by the sampling frequency and the FFT size:

$$
\Delta f = \frac{F_s}{N}
$$

For this system:

$$
\Delta f = \frac{32,\text{MHz}}{65536}
$$

$$
\Delta f = 488.28 \text{ Hz}
$$

Each FFT bin therefore represents **488 Hz of bandwidth**.

This value is required when converting FFT amplitudes into **power spectral density (PSD)**.

---

### PSD conversion

If a bin contains power ($P_{bin}$):

$$
PSD = P_{bin} - 10\log_{10}(\Delta f)
$$

Example:

If a bin contains $-90\,\text{dBm}$:

$$
PSD = -90 - 10\log_{10}(488)
$$

$$
PSD \approx -116.9\, \text{dBm/Hz}
$$

---

## DSP Processing Chain

The signal processing chain implemented in the FPGA is:

```
ADC (12 bit)
   ↓
Polyphase Filter Bank (PFB)
   ↓
Digital Down Converter (DDS + CIC)
   ↓
FFT
   ↓
FFT Accumulator (power averaging)
```

Key characteristics:

| Stage       | Function                             |
| ----------- | ------------------------------------ |
| ADC         | digitizes RF signal                  |
| PFB         | channelization                       |
| DDS+CIC     | frequency translation and decimation |
| FFT         | spectral analysis                    |
| Accumulator | noise reduction via averaging        |

---

## Final Measurement Equation

Combining all steps, the RF input power can be estimated directly from the FFT output:

$$
P_{RF}(dBm) = 20\log_{10} \left( \frac{\sqrt{P[k]}}{N \cdot FS} \right) \cdot C
$$

Where:

| Parameter | Meaning                 |
| --------- | ----------------------- |
| ($P[k]$)    | FFT accumulator output  |
| ($N$)       | FFT size                |
| ($FS$)      | full-scale amplitude    |
| ($C$)       | RF calibration constant |

For this system:

* ($N = 65536$)
* ($FS = 32768$)
* ($C ≈ 39.1\,\text{dB}$)

---
# Notebook Section 3 — Python Implementation

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import glob, re

# ==========================================
# SYSTEM PARAMETERS
# ==========================================
Fs = 32e6
N = 65536

# FFT input is the 16-bit DSP output (not the 12-bit ADC)
N_bits = 16
FS = 2**(N_bits - 1)

# FFT calibration constant (measured)
C_fft = 39.10

print("="*80)
print("SYSTEM PARAMETERS")
print("="*80)
print(f"Sampling frequency: {Fs/1e6:.2f} MHz")
print(f"FFT size: {N}")
print(f"DSP resolution: {N_bits} bits")
print(f"Full-scale: {FS}")
print(f"Frequency resolution: {Fs/N:.3f} Hz")
print(f"FFT calibration constant: {C_fft:.2f} dB")
print("="*80)

# ==========================================
# FIND AND SORT FILES
# ==========================================
data_path = "data/f_300*/x_acc_f_data.txt"
file_paths = glob.glob(data_path)

def extract_power(file):
    m = re.search(r'([+-]?\d+)dBm', file)
    return float(m.group(1)) if m else 0

file_paths = sorted(file_paths, key=extract_power, reverse=True)

labels_list = []
for file in file_paths:
    m = re.search(r'f_\d+([+-]\d+)dBm', file)
    if m:
        labels_list.append(f"{m.group(1)} dBm")
    else:
        labels_list.append(file.split('/')[-2])

print(f"\nFound {len(file_paths)} files:")
for i,(file,label) in enumerate(zip(file_paths,labels_list)):
    print(f"  {i+1}. {label}: {file}")

# ==========================================
# PREPARE PLOTTING
# ==========================================
colors = plt.cm.viridis(np.linspace(0,1,len(file_paths)))
fig, axes = plt.subplots(2,1,figsize=(14,10))

freq = np.fft.fftshift(np.fft.fftfreq(N,1/Fs))

# ==========================================
# PROCESS EACH FILE
# ==========================================
peak_info = []

for i,(file_path,label) in enumerate(zip(file_paths,labels_list)):

    print(f"\nProcessing: {label}")

    P_k = np.loadtxt(file_path)

    # Align spectrum with frequency axis
    #P_k = np.fft.fftshift(P_k)

    # Recover magnitude
    X_mag = np.sqrt(P_k)

    # Recover tone amplitude
    A = X_mag / N

    # Convert to dBFS
    A_dBFS = 20*np.log10(A/FS + 1e-20)

    # PSD
    enbw_hz = 1.5*(Fs/N)
    PSD_dBFS_Hz = A_dBFS - 10*np.log10(enbw_hz)

    # Peak detection
    peak_idx = np.argmax(A_dBFS)

    peak_freq = freq[peak_idx]
    peak_dBFS = A_dBFS[peak_idx]

    # Estimate RF input power using FFT calibration
    pin_est = peak_dBFS + C_fft

    peak_info.append({
        'label': label,
        'pin_est': pin_est,
        'peak_freq': peak_freq,
        'peak_dBFS': peak_dBFS,
        'peak_linear': A[peak_idx]
    })

    # Plot amplitude
    axes[0].plot(freq/1e6,A_dBFS,
                 color=colors[i],
                 label=f"{label} (Pin≈{pin_est:.1f} dBm)")

    # Plot PSD
    axes[1].plot(freq/1e6,PSD_dBFS_Hz,color=colors[i],label=label)

# ==========================================
# FINALIZE PLOTS
# ==========================================
axes[0].set_title(f"Amplitude Spectrum (FFT Accumulator) | Fs = {Fs/1e6:.2f} MHz",
                  fontsize=13,fontweight='bold')
axes[0].set_ylabel("Amplitude [dBFS]",fontsize=12)
axes[0].grid(True,alpha=0.4)
axes[0].set_ylim([None,5])
axes[0].legend(fontsize=9,loc='upper right',ncol=2)
axes[0].set_xlim([freq[0]/1e6,freq[-1]/1e6])

axes[1].set_xlabel("Frequency [MHz]",fontsize=12)
axes[1].set_ylabel("PSD [dBFS/Hz]",fontsize=12)
axes[1].set_title("Power Spectral Density",fontsize=13,fontweight='bold')
axes[1].grid(True,alpha=0.4)
axes[1].legend(fontsize=9,loc='upper right',ncol=2)
axes[1].set_xlim([freq[0]/1e6,freq[-1]/1e6])

plt.tight_layout()
plt.savefig('images/FFT_ACC_AmpdBFS_and_PSD.png',dpi=300,bbox_inches='tight')
plt.show()

# ==========================================
# PRINT SUMMARY TABLE
# ==========================================
print("\n"+"="*80)
print("PEAK ANALYSIS SUMMARY")
print("="*80)
print(f"{'Input Power (est)':<18} {'Peak Freq (MHz)':<18} {'Peak (dBFS)':<15} {'Peak Linear':<15}")
print("-"*80)

for info in peak_info:
    print(f"{info['pin_est']:>8.2f} dBm {info['peak_freq']/1e6:>16.3f} {info['peak_dBFS']:>14.2f} {info['peak_linear']:>14.6f}")

print("="*80)

# ==========================================
# ADDITIONAL ANALYSIS
# ==========================================
input_est=[]
peak_dBFS_vals=[]

for info in peak_info:
    input_est.append(info['pin_est'])
    peak_dBFS_vals.append(info['peak_dBFS'])

fig2,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))

ax1.plot(input_est,peak_dBFS_vals,'o-',markersize=8,linewidth=2,color='blue')
ax1.set_xlabel('Estimated Input Power (dBm)')
ax1.set_ylabel('Peak Amplitude (dBFS)')
ax1.set_title('Peak Amplitude vs Input Power',fontsize=13,fontweight='bold')
ax1.grid(True,alpha=0.3)

if len(input_est)>1:
    input_diffs=np.diff(input_est)
    peak_diffs=np.diff(peak_dBFS_vals)
    gains=peak_diffs/input_diffs

    ax2.plot(input_est[1:],gains,'o-',markersize=8,linewidth=2,color='green')
    ax2.axhline(y=1.0,color='red',linestyle='--',linewidth=2,label='Unity gain')
    ax2.set_xlabel('Input Power (dBm)')
    ax2.set_ylabel('Gain (dB/dB)')
    ax2.set_title('System Gain vs Input Power',fontsize=13,fontweight='bold')
    ax2.legend()
    ax2.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('images/FFT_ACC_Peak_Analysis.png',dpi=300,bbox_inches='tight')
plt.show()

# ==========================================
# ZOOM PLOT
# ==========================================
mid_idx=len(file_paths)//2
P_zoom=np.loadtxt(file_paths[mid_idx])
#P_zoom=np.fft.fftshift(P_zoom)

X_mag_zoom=np.sqrt(P_zoom)
A_zoom=X_mag_zoom/N
A_dBFS_zoom=20*np.log10(A_zoom/FS+1e-20)

zoom_mask=(np.abs(freq)<=1e6)

plt.figure(figsize=(12,6))
plt.plot(freq[zoom_mask]/1e3,A_dBFS_zoom[zoom_mask],linewidth=2,color='blue')
plt.axvline(x=0,color='red',linestyle='--',linewidth=2,alpha=0.5,label='DC')
plt.xlabel('Frequency [kHz]',fontsize=12)
plt.ylabel('Amplitude [dBFS]',fontsize=12)
plt.title(f'Zoomed Spectrum (±1 MHz) - {labels_list[mid_idx]}',
          fontsize=13,fontweight='bold')
plt.grid(True,alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('images/FFT_ACC_Zoom.png',dpi=300,bbox_inches='tight')
plt.show()

print("\nAnalysis complete. Plots saved to images/ directory")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob, re

# ==========================================================
# COMPLETE DSP CHAIN PARAMETERS
# ==========================================================
print("="*80)
print("COMPLETE DSP CHAIN PARAMETERS")
print("="*80)

# Sampling rates
fs_adc = 2048e6
fs_pfb = 256e6
D_cic = 8
fs_cic = fs_pfb / D_cic

# FFT parameters
N_fft = 65536
fs_acc = fs_cic

# FFT numeric full scale (16-bit DSP output)
N_bits_fft = 16
FS = 2**(N_bits_fft - 1)

print(f"ADC rate: {fs_adc/1e6:.0f} MHz")
print(f"PFB output rate: {fs_pfb/1e6:.0f} MHz")
print(f"CIC output rate: {fs_cic/1e6:.0f} MHz")
print(f"FFT size: {N_fft}")
print(f"FFT numeric full scale: {FS}")

# ==========================================================
# DSP SCALING PARAMETERS
# ==========================================================

# Current configuration
QOUT = 6
QPROD = 14

# Reference configuration used during calibration
QOUT_REF = 6
QPROD_REF = 14

# Measured FFT calibration constant for reference config
C_REF = 39.10

# Gain correction
delta_pfb = -6.02*(QOUT - QOUT_REF)
delta_dds = -6.02*(QPROD - QPROD_REF)

C_fft = C_REF + delta_pfb + delta_dds

print("\nCalibration constant:")
print(f"C_fft = {C_fft:.2f} dB")

# ==========================================================
# LOAD DATA
# ==========================================================

data_path = "data/f_300*/x_acc_f_data.txt"
file_paths = glob.glob(data_path)

def extract_power(file):
    m = re.search(r'([+-]?\d+)dBm', file)
    return float(m.group(1)) if m else 0

file_paths = sorted(file_paths, key=extract_power, reverse=True)

labels = []
for f in file_paths:
    labels.append(f"{extract_power(f)} dBm")

print(f"\nFound {len(file_paths)} files")

# ==========================================================
# FREQUENCY AXIS
# ==========================================================

freq = np.fft.fftshift(np.fft.fftfreq(N_fft, 1/fs_acc))

# ==========================================================
# PROCESS DATA
# ==========================================================

colors = plt.cm.viridis(np.linspace(0,1,len(file_paths)))

fig,axes = plt.subplots(3,1,figsize=(14,12))

peak_info=[]

for i,(file,label) in enumerate(zip(file_paths,labels)):

    print("\nProcessing",label)

    P = np.loadtxt(file)

    #P = np.fft.fftshift(P)

    Xmag = np.sqrt(P)

    A = Xmag / N_fft

    A_dBFS = 20*np.log10(A/FS + 1e-20)

    A_dBm = A_dBFS + C_fft

    # PSD
    enbw_bins = 1.5
    enbw = enbw_bins*(fs_acc/N_fft)

    PSD_dBm_Hz = A_dBm - 10*np.log10(enbw)

    peak_idx = np.argmax(A_dBFS)

    peak_freq = freq[peak_idx]
    peak_dBFS = A_dBFS[peak_idx]
    peak_dBm = A_dBm[peak_idx]

    Pin = float(label.split()[0])

    err = peak_dBm - Pin

    peak_info.append((Pin,peak_dBm,err,peak_dBFS))

    print(f"Peak {peak_dBFS:.2f} dBFS = {peak_dBm:.2f} dBm")
    print(f"Error {err:+.2f} dB")

    axes[0].plot(freq/1e6,A_dBm,color=colors[i],label=label)
    axes[1].plot(freq/1e6,A_dBFS,color=colors[i])
    axes[2].plot(freq/1e6,PSD_dBm_Hz,color=colors[i])

# ==========================================================
# PLOTS
# ==========================================================

axes[0].set_ylabel("Amplitude [dBm]")
axes[0].set_title("Calibrated Spectrum")
axes[0].grid()
axes[0].legend()

axes[1].set_ylabel("Amplitude [dBFS]")
axes[1].set_title("Spectrum (dBFS)")
axes[1].grid()

axes[2].set_ylabel("PSD [dBm/Hz]")
axes[2].set_xlabel("Frequency [MHz]")
axes[2].set_title("Power Spectral Density")
axes[2].grid()

plt.tight_layout()
plt.show()

# ==========================================================
# CALIBRATION VERIFICATION
# ==========================================================

print("\n"+"="*80)
print("CALIBRATION VERIFICATION")
print("="*80)

Pin=[]
Pout=[]
err=[]

for p in peak_info:

    Pin.append(p[0])
    Pout.append(p[1])
    err.append(p[2])

    print(f"{p[0]:>6.1f} dBm   {p[1]:>7.2f} dBm   err {p[2]:>6.2f} dB")

print("\nStatistics")

print("Mean error:",np.mean(err))
print("Std dev:",np.std(err))
print("Max error:",np.max(np.abs(err)))

# ==========================================================
# POWER RECOVERY PLOT
# ==========================================================

plt.figure(figsize=(8,6))

plt.plot(Pin,Pout,'o',label="Recovered")
plt.plot(Pin,Pin,'k--',label="Ideal")

plt.xlabel("Input power (dBm)")
plt.ylabel("Recovered power (dBm)")

plt.title("RF Power Recovery")

plt.legend()
plt.grid()

plt.show()

# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n"+"="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"""
DSP Chain

ADC → PFB → DDS → CIC → FFT

Calibration constant

C_fft = {C_fft:.2f} dB

RF power estimation

P_RF(dBm) = P_FFT(dBFS) + C_fft

Accuracy

σ ≈ {np.std(err):.3f} dB
""")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob, re

# ==========================================================
# SYSTEM PARAMETERS
# ==========================================================

fs_adc = 2048e6
fs_pfb = 256e6
D_cic = 8
fs_cic = fs_pfb / D_cic

N_fft = 65536
fs_acc = fs_cic

N_bits_adc = 16
FS = 2**(N_bits_adc-1)

print("ADC full scale:", FS)
print("FFT size:", N_fft)

# ==========================================================
# LOAD FFT ACCUMULATOR FILES
# ==========================================================

data_path = "data/f_300*/x_acc_f_data.txt"
files = glob.glob(data_path)

def extract_power(f):
    m = re.search(r'([+-]?\d+)dBm',f)
    return float(m.group(1))

files = sorted(files,key=extract_power,reverse=True)

freq = np.fft.fftshift(np.fft.fftfreq(N_fft,1/fs_acc))

measurements = []

# ==========================================================
# PROCESS FFT DATA
# ==========================================================

for f in files:

    Pin = extract_power(f)

    # load accumulated power spectrum
    P = np.loadtxt(f)

    # recover magnitude
    Xmag = np.sqrt(P)

    # recover time-domain amplitude
    A = Xmag / N_fft

    # convert to dBFS
    A_dBFS = 20*np.log10(A/FS + 1e-20)

    peak_idx = np.argmax(A_dBFS)
    peak_dBFS = A_dBFS[peak_idx]

    measurements.append((Pin,peak_dBFS,f))

    print(f"{Pin:+4.0f} dBm  ->  {peak_dBFS:7.2f} dBFS")

# ==========================================================
# FFT CALIBRATION
# ==========================================================

Pin = np.array([m[0] for m in measurements])
Pfft = np.array([m[1] for m in measurements])

# fit calibration constant
coeff = np.polyfit(Pfft,Pin,1)

slope = coeff[0]
C_fft = coeff[1]

print("\nFFT CALIBRATION")
print("Slope:",slope)
print("Calibration constant:",C_fft)

# ==========================================================
# VERIFY CALIBRATION
# ==========================================================

Pin_est = slope*Pfft + C_fft
error = Pin_est - Pin

print("\nCALIBRATION VERIFICATION")
print("Mean error:",np.mean(error))
print("Std dev:",np.std(error))

# ==========================================================
# PLOT RESULTS
# ==========================================================

plt.figure(figsize=(10,6))

plt.plot(Pin,Pin_est,'o',label="Recovered power")
plt.plot(Pin,Pin,'k--',label="Ideal")

plt.xlabel("True input power (dBm)")
plt.ylabel("Recovered power (dBm)")
plt.title("FFT Accumulator Calibration")

plt.legend()
plt.grid()

plt.show()

# ==========================================================
# FUNCTION TO ESTIMATE RF INPUT POWER
# ==========================================================

def estimate_rf_power(P_fft_bin_dBFS):
    """
    Convert FFT bin amplitude (dBFS) to RF input power (dBm)
    using calibrated FFT accumulator constant.
    """
    return slope*P_fft_bin_dBFS + C_fft


The FFT calibration constant depends on the programmable scaling
parameters of the DSP chain (PFB QOUT and DDS+CIC QPROD).

A reference calibration was obtained for:

    QOUT = 6
    QPROD = 14

For other configurations the calibration constant can be corrected
analytically because both parameters correspond to power-of-two
scalings.

In [ ]:
# ==========================================
# DSP SCALING PARAMETERS
# ==========================================

QOUT = 6
QPROD = 14

# reference configuration used during calibration
QOUT_REF = 6
QPROD_REF = 14
C_REF = 39.10

# compute gain correction
delta_pfb = -6.02 * (QOUT - QOUT_REF)
delta_dds = -6.02 * (QPROD - QPROD_REF)

C_fft = C_REF + delta_pfb + delta_dds

print("Effective calibration constant:", C_fft)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# System Parameters
# -----------------------------
Fs = 32e6               # Sampling frequency
N = 65536               # FFT size
N_bits = 12             # ADC resolution
FS = 2**(N_bits - 1)    # Full-scale amplitude (8192)

# -----------------------------
# Load FFT power data
# -----------------------------
# P[k] = averaged |X[k]|^2
P = np.loadtxt("data/f_300+6dBm/x_acc_f_data.txt")

# -----------------------------
# Recover FFT magnitude
# -----------------------------
# |X[k]| = sqrt(P[k])
X_mag = np.sqrt(P)

# -----------------------------
# Recover real tone amplitude
# -----------------------------
# Since FFT is unscaled:
# |X[k0]| = N * A
# A = |X[k0]| / N
A = X_mag / N

# -----------------------------
# Normalize to Full-Scale
# -----------------------------
A_FS = A / FS

# Convert to dBFS
A_dBFS = 20 * np.log10(A_FS + 1e-20)

# -----------------------------
# Frequency Axis
# -----------------------------
# Spectrum is already centered (tone at DC)
freq = np.linspace(-Fs/2, Fs/2, N)

# -----------------------------
# Plot Amplitude Spectrum
# -----------------------------
plt.figure(figsize=(10,4))
plt.plot(freq, A_dBFS)
plt.title("Amplitude Spectrum (dBFS)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude (dBFS)")
plt.grid(True)
plt.tight_layout()
plt.show()
print("P at DC:", P[N//2])
print("sqrt(P):", np.sqrt(P[N//2]))
print("A_dBFS at DC:", A_dBFS[N//2])

---

# Notebook Section 4 — PSD Derivation

Now the correct PSD derivation.

---

## (1) True signal power

Since:

$$
\begin{equation}
|X[k]| = N A
\end{equation}
$$

Then:

$$
\begin{equation}
|X[k]|^2 = N^2 A^2
\end{equation}
$$

To recover actual signal power:

$$
\begin{equation}
A^2 = \frac{P[k]}{N^2}
\end{equation}
$$

---

## (2) Convert to Power Spectral Density

Each FFT bin represents:

$$
\begin{equation}
\Delta f = \frac{Fs}{N}
\end{equation}
$$

So PSD is:

$$
\begin{equation}
PSD[k] = \frac{A^2}{\Delta f}
\end{equation}
$$

Combining:

$$
\begin{equation}
PSD[k] = \frac{P[k]}{N^2 \Delta f}
\end{equation}
$$

---

## (3) Convert to dBFS/Hz

Normalize power to $\text{FS}^2$:

$$
\begin{equation}
PSD_{FS} = \frac{PSD}{FS^2}
\end{equation}
$$

Then:

$$
\begin{equation}
10\log_{10}(PSD_{FS})
\end{equation}
$$

---

# Notebook Section 5 — PSD Code

In [ ]:
# -----------------------------
# Frequency resolution
# -----------------------------
df = Fs / N

# -----------------------------
# True power per bin
# -----------------------------
# A^2 = P / N^2
# PSD = A^2 / df
PSD = P / (N**2 * df)

# Normalize to FS power
PSD_FS = PSD / (FS**2)

PSD_dBFS_per_Hz = 10 * np.log10(PSD_FS + 1e-20)

# -----------------------------
# Plot PSD
# -----------------------------
plt.figure(figsize=(10,4))
plt.plot(freq, PSD_dBFS_per_Hz)
plt.title("Power Spectral Density (dBFS/Hz)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (dBFS/Hz)")
plt.grid(True)
plt.tight_layout()
plt.show()

---

# Final Summary

---

### Why divide by N?

Because the FFT is unscaled:

$$
\begin{equation}
X[k] = \sum x[n]
\end{equation}
$$

So coherent tones grow proportionally to $N$.

---

### Why divide by $N^2$ in PSD?

Because:

$$
\begin{equation}
|X|^2 = N^2 A^2
\end{equation}
$$

We must remove FFT gain before computing physical power.

---

### Why $20\log_{10}$ for amplitude and $10\log_{10}$ for PSD?

Because:

* Amplitude is proportional to voltage $\rightarrow 20\log_{10}$
* PSD is power $\rightarrow 10\log_{10}$

---

# What this notebook now guarantees

* Correct amplitude recovery
* Correct dBFS normalization
* Correct PSD scaling
* Physically interpretable results
---